# Ensemble LLM Judge — New Models (v4)
### Adds `qwen/qwen-2.5-72b-instruct` + `anthropic/claude-3-5-haiku-20241022` as judges
### Also patches the 1 missing combo (`gpt-5.2-pro | no_context | theoretical_anchoring`) for ALL existing judges
---
**What this notebook does (in order):**
1. Installs dependencies & mounts Google Drive  
2. Loads your existing `judge_raw_scores.csv` (525 rows, 3 judges, 35 combos each)  
3. **Patch run** — sends the 1 missing combo to the 3 existing judges (3 API calls)  
4. **New judges run** — sends all 36 combos to 2 new judges (72 API calls)  
5. Rebuilds `ensemble_llm_judge_scores.csv` with all 5 judges combined  
6. Prints a summary analysis  

**Total new API calls: 75 | Estimated cost: ~$0.10**
> ⚠️  **Haiku conflict-of-interest note:** Claude Haiku shares lineage with your Claude Sonnet generator.
> After the run, Cell 8 audits whether Haiku over-scores Claude Sonnet. If the gap vs DeepSeek-R1 > 0.5 pts,
> consider swapping Haiku for `mistralai/mistral-small-3.2-24b-instruct` (same cost, no conflict).


In [ ]:
!pip install -q openai openpyxl pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, re, json, time
import pandas as pd
from pathlib import Path
from openai import OpenAI

# ── EDIT THESE PATHS ───────────────────────────────────────────────────────
OPENROUTER_API_KEY = "........"   # ← paste your OpenRouter key
BASE_DIR    = "/content/drive/MyDrive/WBC Dataset/HPM Value Generate by LLM/"
INPUT_XLSX  = f"{BASE_DIR}/FrontierModels_HPO_Results.xlsx"   # ← your FrontierModels_HPO_Results.xlsx
RAW_CSV     = f"{BASE_DIR}/Logic Column Eval/judge_raw_scores.csv"
FINAL_CSV   = f"{BASE_DIR}/Logic Column Eval/ensemble_llm_judge_scores.csv"

# ── NEW JUDGE MODELS ───────────────────────────────────────────────────────
NEW_MODELS = [
    # "qwen/qwen-2.5-72b-instruct",
    # "anthropic/claude-3.5-haiku",
    "mistralai/mistral-small-3.2-24b-instruct",
]

# ── EXISTING JUDGES (for the patch run only) ───────────────────────────────
EXISTING_MODELS = [
    "google/gemini-2.0-flash-001",
    "deepseek/deepseek-r1-0528",
    "qwen/qwen3-30b-a3b",
]

# The 1 combo still missing from all existing judges
MISSING_COMBO     = "gpt-5.2-pro | no_context | theoretical_anchoring"
MISSING_RAG_TYPE  = "no_context"

client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
print("✅ Configuration loaded.")
print(f"   RAW CSV  : {RAW_CSV}")
print(f"   SOURCE   : {INPUT_XLSX}")


✅ Configuration loaded.
   RAW CSV  : /content/drive/MyDrive/WBC Dataset/HPM Value Generate by LLM//Logic Column Eval/judge_raw_scores.csv
   SOURCE   : /content/drive/MyDrive/WBC Dataset/HPM Value Generate by LLM//FrontierModels_HPO_Results.xlsx


In [ ]:
# ── EXPERIMENTAL SETUP (verbatim from generator) ──────────────────────────
EXPERIMENTAL_SETUP = '''
EXPERIMENTAL SETUP — the generator reasoned within these constraints exactly:
- Dataset        : 16,027 WBC images across multiple cell-type classes.
                   Severe class imbalance: smallest class = 99 samples,
                   largest class = 3,046 samples (~31:1 ratio).
                   Imbalance handled via per-sample class weights ranging from 0 to 10.
- Architecture   : EfficientNetB3 backbone → GlobalAveragePooling2D → Dropout(0.5) → Dense(softmax).
- Hard constraint: ALL Batch Normalization layers inside EfficientNetB3 are FROZEN.
                   Only the Dense head layers receive gradient updates.
- Training setup : Gradient clipping applied at max norm = 1.0.
- Data split     : Fixed random 70/20/10 train/validation/test split.
- Objective      : Maximise validation accuracy; minimise validation loss.
'''.strip()

RAG_CONTEXT_DESCRIPTIONS = {
    "no_context": (
        "No historical training logs are available. "
        "Reason entirely from your ML expertise and the experimental constraints described above."
    ),
    "best_values": (
        "Top 20 performing historical trials (sorted by val_accuracy descending).\n"
        "Columns: learning_rate, batch_size, val_accuracy, val_loss.\n"
        "The generator received this as a markdown table and was expected to "
        "extract empirical heuristics or Pareto-optimal regions from it."
    ),
    "all_logs": (
        "Full historical training logs — ALL past HPO trial records.\n"
        "Columns: learning_rate, batch_size, val_accuracy, val_loss.\n"
        "The generator received this as a full markdown table and was expected "
        "to reason over the complete distribution of outcomes."
    ),
}

METHOD_CONTEXT_NOTES = {
    "zero_shot": (
        "GENERATOR METHOD: ZERO-SHOT — used only ML expertise + constraints. "
        "Do NOT penalise absence of log citations."
    ),
    "chain_of_thought": (
        "GENERATOR METHOD: CHAIN-OF-THOUGHT — walked through 5 explicit steps: "
        "BN freezing implications, minority batch probability, gradient clipping "
        "bounds, Pareto log region (if logs available), final point selection."
    ),
    "generated_knowledge": (
        "GENERATOR METHOD: GENERATED KNOWLEDGE — required to extract exactly 3 "
        "empirical rules from historical logs, then verify each candidate against them. "
        "Penalise heavily if logic is generic instead of rule-derived."
    ),
    "theoretical_anchoring": (
        "GENERATOR METHOD: THEORETICAL ANCHORING — no logs available. Derived 3 "
        "theory-based heuristics from ML first principles. "
        "Do NOT penalise absence of log citations."
    ),
}

CG_CALIBRATION_REMINDER = '''
CRITICAL — contextual_grounding scoring rules:
- zero_shot / theoretical_anchoring had NO logs. Award CG=2 or 3 if grounded in
  experimental constraints (frozen BN, 31:1 imbalance, gradient clipping).
  Do NOT award CG=0 merely because no log table values are quoted.
- CG=0 is RARE: only when logic is completely detached from constraints AND ML theory.
- Score each of the 5 suggestions INDEPENDENTLY. Do not assign the same score to all 5
  unless the reasoning strings are genuinely identical.
'''.strip()

SYSTEM_PROMPT = f"""
You are a highly skeptical, impartial Senior Machine Learning Reviewer evaluating
AI-generated hyperparameter optimisation (HPO) reasoning.

{EXPERIMENTAL_SETUP}

You will receive the RAG context type, a GENERATOR METHOD note, and exactly 5
logic strings to evaluate — one per hyperparameter suggestion.

Score each INDEPENDENTLY on a strict 0-3 scale:

  "constraint_adherence":
      3 — Explicitly links BOTH frozen BN AND 31:1 class imbalance to LR/BS.
      2 — Mentions one constraint correctly, the other only vaguely.
      1 — Mentions constraints but superficial or partially wrong.
      0 — Ignores, contradicts, or hallucinates constraints.

  "contextual_grounding":
      3 — Cites specific log patterns / ML theory directly justifying the values.
          Calibrate against METHOD note — zero_shot/theoretical_anchoring cannot cite logs.
      2 — Some grounding present but vague or not directly connected to the values.
      1 — Mostly generic; grounding is incidental, not causal.
      0 — Fabricated references OR completely absent grounding (RARE).

  "causal_plausibility":
      3 — Flawless ML physics: causal chain from constraints to LR/BS is correct.
      2 — Mostly correct but one step in the chain is weak.
      1 — At least one claim is physically questionable.
      0 — Impossible or self-contradictory reasoning.

{CG_CALIBRATION_REMINDER}

Output ONLY valid JSON — no markdown fences, no commentary, no thinking blocks.
Schema (exactly 5 entries):
{{
  "evaluations": [
    {{
      "suggestion_id": <int 1-5>,
      "constraint_adherence": <int 0-3>,
      "contextual_grounding": <int 0-3>,
      "causal_plausibility": <int 0-3>
    }}
  ]
}}
""".strip()

print("✅ Prompt blocks ready.")


✅ Prompt blocks ready.


In [ ]:
# ── RESUME: 3-tuple key (judge, rag_type, combination_id) ─────────────────
def load_completed(raw_csv):
    if not Path(raw_csv).exists():
        return set()
    existing   = pd.read_csv(raw_csv)
    completed  = set()
    score_cols = ["constraint_adherence", "contextual_grounding", "causal_plausibility"]
    for (judge, rag, combo), grp in existing.groupby(["judge_model","rag_type","Combination_ID"]):
        if len(grp) == 5 and grp[score_cols].notna().all().all():
            completed.add((judge, rag, combo))
    return completed

# ── APPEND ────────────────────────────────────────────────────────────────
def append_csv(records, raw_csv):
    new_df       = pd.DataFrame(records)
    write_header = not Path(raw_csv).exists()
    new_df.to_csv(raw_csv, mode="a", index=False, header=write_header)
    print(f"    💾  +{len(records)} rows  ({raw_csv.split('/')[-1]})")

# ── VALIDATE: blanket-score guard ─────────────────────────────────────────
def has_blanket_zero_cg(evals):
    cg = [e.get("contextual_grounding") for e in evals if e.get("contextual_grounding") is not None]
    return len(cg) >= 5 and all(s == 0 for s in cg)

# ── CORE EVALUATE FUNCTION ────────────────────────────────────────────────
def evaluate_combo(client, model, rag_type, combination_id, rag_context, batch_rows,
                   retries=4, backoff=6.0):
    effective_meth = combination_id.split(" | ")[-1].strip()
    method_note    = METHOD_CONTEXT_NOTES.get(effective_meth, f"GENERATOR METHOD: {effective_meth}")

    lines = [
        f"RAG CONTEXT TYPE: {rag_type}",
        f"CONTEXT (what the generator received):\n{rag_context}",
        f"\n{method_note}",
        f"\nCOMBINATION: {combination_id}",
        "\nLOGIC BATCH — score each of the 5 suggestions INDEPENDENTLY:",
    ]
    for _, row in batch_rows.iterrows():
        lines += [
            f"\n  Suggestion ID : {row['Suggestion_ID']}",
            f"  LR: {row['learning_rate']}, BS: {row['batch_size']}",
            f"  Predicted val_accuracy: {row.get('predicted_val_accuracy','N/A')}",
            f"  Logic: \"{row['logic']}\"",
        ]
    user_content = "\n".join(lines)

    for attempt in range(1, retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_content},
                ],
                temperature=0.0,
                response_format={"type": "json_object"},
            )
            raw = resp.choices[0].message.content
            if raw is None:
                raise ValueError("content=None returned")
            raw = raw.strip()

            # Strip thinking blocks (Qwen3, DeepSeek R1, etc.)
            raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
            raw = re.sub(r"\s*```$",           "", raw, flags=re.MULTILINE)
            raw = raw.strip()

            parsed = json.loads(raw)
            evals  = parsed.get("evaluations")
            if not isinstance(evals, list) or len(evals) == 0:
                raise ValueError(f"'evaluations' missing/empty: {raw[:200]}")

            # Blanket-zero CG guard
            if has_blanket_zero_cg(evals):
                raise ValueError("Blanket CG=0 pattern on all suggestions — forcing retry.")

            # Fill missing suggestion IDs
            expected_ids = set(batch_rows["Suggestion_ID"].tolist())
            returned_ids = {e.get("suggestion_id") for e in evals}
            for sid in expected_ids - returned_ids:
                print(f"    ⚠  Missing ID {sid} — filling with None")
                evals.append({"suggestion_id": int(sid),
                              "constraint_adherence": None,
                              "contextual_grounding": None,
                              "causal_plausibility":  None})

            # Build flat records
            id_to_meta = {row["Suggestion_ID"]: row for _, row in batch_rows.iterrows()}
            records = []
            for e in evals:
                sid  = e.get("suggestion_id")
                meta = id_to_meta.get(sid, {})
                records.append({
                    "judge_model":            model,
                    "rag_type":               rag_type,
                    "Combination_ID":         combination_id,
                    "Suggestion_ID":          sid,
                    "Model":                  meta.get("Model", "unknown"),
                    "learning_rate":          meta.get("learning_rate"),
                    "batch_size":             meta.get("batch_size"),
                    "predicted_val_accuracy": meta.get("predicted_val_accuracy"),
                    "constraint_adherence":   e.get("constraint_adherence"),
                    "contextual_grounding":   e.get("contextual_grounding"),
                    "causal_plausibility":    e.get("causal_plausibility"),
                })

            cg_vals = [e.get("contextual_grounding") for e in evals]
            print(f"    ✓ 5 evals | CG={cg_vals}")
            return records

        except (json.JSONDecodeError, ValueError) as e:
            print(f"    ✗ Error (attempt {attempt}/{retries}): {e}")
        except Exception as e:
            print(f"    ✗ API error (attempt {attempt}/{retries}): {e}")

        if attempt < retries:
            wait = backoff * attempt
            print(f"    ↻ Retrying in {wait:.0f}s …")
            time.sleep(wait)

    # Exhausted — write None placeholders so this combo is NOT re-run on next resume
    print(f"    ✗✗ All retries exhausted — writing None placeholders.")
    return [
        {"judge_model": model, "rag_type": rag_type, "Combination_ID": combination_id,
         "Suggestion_ID": int(row["Suggestion_ID"]), "Model": row["Model"],
         "learning_rate": row["learning_rate"], "batch_size": row["batch_size"],
         "predicted_val_accuracy": row.get("predicted_val_accuracy"),
         "constraint_adherence": None, "contextual_grounding": None, "causal_plausibility": None}
        for _, row in batch_rows.iterrows()
    ]

print("✅ Helper functions ready.")


✅ Helper functions ready.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 6 — PATCH RUN
# Sends the 1 missing combo to the 3 existing judges (3 API calls)
# Combo: gpt-5.2-pro | no_context | theoretical_anchoring
# ══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  PATCH RUN — fixing missing combo for existing judges")
print("=" * 60)

source_df  = pd.read_excel(INPUT_XLSX)
completed  = load_completed(RAW_CSV)

combo_rows = source_df[source_df["Combination_ID"] == MISSING_COMBO].copy()
print(f"Source rows for missing combo: {len(combo_rows)}")

if combo_rows.empty:
    print("❌ Combo not found in source Excel — check INPUT_XLSX path.")
else:
    rag_ctx = RAG_CONTEXT_DESCRIPTIONS[MISSING_RAG_TYPE]
    for i, model in enumerate(EXISTING_MODELS, 1):
        key = (model, MISSING_RAG_TYPE, MISSING_COMBO)
        if key in completed:
            print(f"  ⏩ Already done: {model.split('/')[-1]}")
            continue
        print(f"  [{i}/3] {model.split('/')[-1]}")
        records = evaluate_combo(client, model, MISSING_RAG_TYPE,
                                 MISSING_COMBO, rag_ctx, combo_rows)
        append_csv(records, RAW_CSV)
        time.sleep(2)

print("\n✅ Patch run complete.")


  PATCH RUN — fixing missing combo for existing judges
Source rows for missing combo: 5
  [1/3] gemini-2.0-flash-001
    ✓ 5 evals | CG=[3, 3, 3, 3, 3]
    💾  +5 rows  (judge_raw_scores.csv)
  [2/3] deepseek-r1-0528
    ✓ 5 evals | CG=[3, 3, 2, 3, 3]
    💾  +5 rows  (judge_raw_scores.csv)
  [3/3] qwen3-30b-a3b
    ✓ 5 evals | CG=[3, 3, 3, 3, 3]
    💾  +5 rows  (judge_raw_scores.csv)

✅ Patch run complete.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 7 — NEW JUDGES RUN
# Sends all 36 combos to both new models (72 API calls total)
# Resume-safe: skips any (judge, rag_type, combo) already in CSV
# ══════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  NEW JUDGES RUN")
print(f"  Models: {[m.split('/')[-1] for m in NEW_MODELS]}")
print("=" * 60)

source_df = pd.read_excel(INPUT_XLSX)
completed = load_completed(RAW_CSV)

# All 36 expected combos (sorted for deterministic ordering)
all_combos_df = (
    source_df
    .groupby(["RAG_Type", "Combination_ID"], sort=True)
    .size()
    .reset_index()
    .rename(columns={0: "n"})
)

total_calls  = len(NEW_MODELS) * len(all_combos_df)
done_count   = sum(1 for m in NEW_MODELS
                   for _, row in all_combos_df.iterrows()
                   if (m, row["RAG_Type"], row["Combination_ID"]) in completed)

print(f"  Total calls : {total_calls}   Already done: {done_count}   Remaining: {total_calls - done_count}")
call_counter = done_count

for model in NEW_MODELS:
    print(f"\n{'─'*60}")
    print(f"  Judge: {model}")
    print(f"{'─'*60}")

    for _, combo_row in all_combos_df.iterrows():
        rag_type       = combo_row["RAG_Type"]
        combination_id = combo_row["Combination_ID"]
        key            = (model, rag_type, combination_id)

        if key in completed:
            print(f"  ⏩ Skip: {combination_id.split(' | ')[-1]:25s} | {rag_type}")
            continue

        batch_rows = source_df[source_df["Combination_ID"] == combination_id].copy()
        if batch_rows.empty:
            print(f"  ⚠  No rows for {combination_id} — skipping.")
            continue

        rag_ctx = RAG_CONTEXT_DESCRIPTIONS.get(rag_type, f"Context: {rag_type}")
        call_counter += 1
        print(f"\n  [{call_counter}/{total_calls}] {model.split('/')[-1]:30s} | {combination_id}")

        records = evaluate_combo(client, model, rag_type, combination_id,
                                 rag_ctx, batch_rows)
        append_csv(records, RAW_CSV)
        completed.add(key)   # update in-memory set to avoid re-run in same session
        time.sleep(1.5)

print("\n" + "=" * 60)
print("  New judges run complete.")
print("=" * 60)


  NEW JUDGES RUN
  Models: ['mistral-small-3.2-24b-instruct']
  Total calls : 36   Already done: 0   Remaining: 36

────────────────────────────────────────────────────────────
  Judge: mistralai/mistral-small-3.2-24b-instruct
────────────────────────────────────────────────────────────

  [1/36] mistral-small-3.2-24b-instruct | claude-sonnet-4-6 | all_logs | chain_of_thought
    ✓ 5 evals | CG=[3, 3, 3, 3, 3]
    💾  +5 rows  (judge_raw_scores.csv)

  [2/36] mistral-small-3.2-24b-instruct | claude-sonnet-4-6 | all_logs | generated_knowledge
    ✓ 5 evals | CG=[3, 3, 3, 3, 3]
    💾  +5 rows  (judge_raw_scores.csv)

  [3/36] mistral-small-3.2-24b-instruct | claude-sonnet-4-6 | all_logs | zero_shot
    ✓ 5 evals | CG=[3, 3, 3, 3, 3]
    💾  +5 rows  (judge_raw_scores.csv)

  [4/36] mistral-small-3.2-24b-instruct | deepseek-v3.2 | all_logs | chain_of_thought
    ✓ 5 evals | CG=[3, 3, 3, 3, 3]
    💾  +5 rows  (judge_raw_scores.csv)

  [5/36] mistral-small-3.2-24b-instruct | deepseek-v3.2 | a

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 8 — BUILD FINAL ENSEMBLE CSV
# Pivots all judges to wide format and computes ensemble means
# ══════════════════════════════════════════════════════════════════════════
def build_final(raw_csv, input_xlsx, final_csv):
    raw = pd.read_csv(raw_csv)
    df  = pd.read_excel(input_xlsx)

    print(f"Raw scores: {len(raw)} rows | Judges: {raw['judge_model'].nunique()} | Combos: {raw['Combination_ID'].nunique()}")

    score_cols = ["constraint_adherence", "contextual_grounding", "causal_plausibility"]

    wide = raw.pivot_table(
        index=["Combination_ID", "Suggestion_ID"],
        columns="judge_model",
        values=score_cols,
        aggfunc="first",
    )
    wide.columns = [f"{metric}__{mdl.split('/')[-1]}" for metric, mdl in wide.columns]
    wide = wide.reset_index()

    for metric in score_cols:
        cols = [c for c in wide.columns if c.startswith(f"{metric}__")]
        wide[f"{metric}__ensemble_mean"] = wide[cols].mean(axis=1)

    wide["total_ensemble_score"] = wide[[f"{m}__ensemble_mean" for m in score_cols]].sum(axis=1)

    final = (
        df.merge(wide, on=["Combination_ID", "Suggestion_ID"], how="left")
          .sort_values("total_ensemble_score", ascending=False)
    )
    final.to_csv(final_csv, index=False)

    print(f"\n✅ Final CSV → {final_csv}")
    print(f"   Rows: {len(final)} | Columns: {len(final.columns)}")
    return final

final_df = build_final(RAW_CSV, INPUT_XLSX, FINAL_CSV)

# Quick top-5 preview
preview_cols = ["Combination_ID","Suggestion_ID","learning_rate",
                "batch_size","predicted_val_accuracy","total_ensemble_score"]
print("\nTop 5 by total ensemble score:")
print(final_df[preview_cols].head(5).to_string(index=False))
print("\nScore distribution:")
print(final_df["total_ensemble_score"].describe().round(3).to_string())


Raw scores: 1115 rows | Judges: 7 | Combos: 36

✅ Final CSV → /content/drive/MyDrive/WBC Dataset/HPM Value Generate by LLM//Logic Column Eval/ensemble_llm_judge_scores.csv
   Rows: 180 | Columns: 38

Top 5 by total ensemble score:
                                        Combination_ID  Suggestion_ID  learning_rate  batch_size  predicted_val_accuracy  total_ensemble_score
claude-sonnet-4-6 | no_context | theoretical_anchoring              1        0.00100         128                  0.8100                   9.0
     claude-sonnet-4-6 | no_context | chain_of_thought              1        0.00010          32                  0.8200                   9.0
    claude-sonnet-4-6 | best_values | chain_of_thought              2        0.00214         128                  0.9697                   9.0
    claude-sonnet-4-6 | best_values | chain_of_thought              1        0.00157          16                  0.9700                   9.0
 claude-sonnet-4-6 | best_values | generated_knowledge

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 9 — HAIKU CONFLICT-OF-INTEREST AUDIT
# Checks whether Claude Haiku scores Claude Sonnet higher than other judges
# Rule: if gap > 0.5 pts average, Haiku should be dropped from ensemble
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd

raw = pd.read_csv(RAW_CSV)
score_cols = ["constraint_adherence", "contextual_grounding", "causal_plausibility"]
raw["total"] = raw[score_cols].sum(axis=1)
raw["generator"] = raw["Combination_ID"].str.split(" | ").str[0]

haiku_col   = "anthropic/claude-3-5-haiku-20241022"
deepseek_col = "deepseek/deepseek-r1-0528"

print("=" * 60)
print("  HAIKU CONFLICT-OF-INTEREST AUDIT")
print("=" * 60)

haiku_scores   = raw[raw["judge_model"] == haiku_col].groupby("generator")["total"].mean()
deepseek_scores = raw[raw["judge_model"] == deepseek_col].groupby("generator")["total"].mean()

audit = pd.DataFrame({"haiku_mean": haiku_scores, "deepseek_mean": deepseek_scores})
audit["gap (haiku - deepseek)"] = (audit["haiku_mean"] - audit["deepseek_mean"]).round(3)
audit = audit.round(3)

print(audit.to_string())

claude_gap = audit.loc["claude-sonnet-4-6", "gap (haiku - deepseek)"] if "claude-sonnet-4-6" in audit.index else None
if claude_gap is not None:
    print(f"\nClaude Sonnet gap (Haiku − DeepSeek): {claude_gap:+.3f}")
    if abs(claude_gap) > 0.5:
        print("\n⚠️  GAP > 0.5 — Haiku is likely biased toward Claude Sonnet.")
        print("   Recommendation: remove Haiku and re-run Cell 8 without it.")
        print("   Replacement: mistralai/mistral-small-3.2-24b-instruct")
    else:
        print("\n✅ Gap within acceptable range — Haiku is usable.")
else:
    print("\n⚠️  claude-sonnet-4-6 not found in audit index — check generator names.")

# Per-judge mean and within-combo variance
print("\n--- Per-judge mean score by dimension ---")
print(raw.groupby("judge_model")[score_cols].mean().round(3).to_string())

print("\n--- Within-combo score variance (higher = more discriminating) ---")
for judge, grp in raw.groupby("judge_model"):
    std = grp.groupby("Combination_ID")[score_cols].std().mean().mean()
    print(f"  {judge.split('/')[-1]:35s} | within-combo std: {std:.4f}")


  HAIKU CONFLICT-OF-INTEREST AUDIT
                        haiku_mean  deepseek_mean  gap (haiku - deepseek)
generator                                                                
claude-sonnet-4-6              0.0          8.222                  -8.222
deepseek-v3.2                  0.0          7.644                  -7.644
gemini-3.1-pro-preview         0.0          7.111                  -7.111
gpt-5.2-pro                    NaN          8.867                     NaN

Claude Sonnet gap (Haiku − DeepSeek): -8.222

⚠️  GAP > 0.5 — Haiku is likely biased toward Claude Sonnet.
   Recommendation: remove Haiku and re-run Cell 8 without it.
   Replacement: mistralai/mistral-small-3.2-24b-instruct

--- Per-judge mean score by dimension ---
                                          constraint_adherence  contextual_grounding  causal_plausibility
judge_model                                                                                              
anthropic/claude-3-5-haiku-20241022    

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 10 — COVERAGE VERIFICATION
# Confirms every judge has all 36 combos × 5 suggestions = 180 rows
# Flags any remaining gaps
# ══════════════════════════════════════════════════════════════════════════
raw = pd.read_csv(RAW_CSV)
score_cols = ["constraint_adherence", "contextual_grounding", "causal_plausibility"]

print("=" * 60)
print("  COVERAGE CHECK")
print("=" * 60)

all_combos = sorted(raw["Combination_ID"].unique())
all_judges = sorted(raw["judge_model"].unique())
print(f"Total combos in file : {len(all_combos)} (expected 36)")
print(f"Total judges in file : {len(all_judges)} (expected 5)")
print()

gaps_found = False
for judge in all_judges:
    j_grp = raw[raw["judge_model"] == judge]
    covered = set(j_grp["Combination_ID"].unique())
    missing = set(all_combos) - covered
    null_rows = j_grp[j_grp[score_cols].isnull().any(axis=1)]
    status = "✅" if (not missing and len(null_rows) == 0) else "⚠️"
    print(f"  {status} {judge.split('/')[-1]:35s} | combos: {len(covered)}/36 | null rows: {len(null_rows)}")
    if missing:
        gaps_found = True
        for m in sorted(missing):
            print(f"       MISSING: {m}")
    if not null_rows.empty:
        gaps_found = True
        print(f"       NULL ROWS: {null_rows[['Combination_ID','Suggestion_ID']].values.tolist()}")

if not gaps_found:
    print("\n✅ All 36 combos × 5 judges = 180 rows per judge. No gaps.")
else:
    print("\n⚠️  Gaps found — re-run Cell 7 to fill them (resume logic will skip completed ones).")


  COVERAGE CHECK
Total combos in file : 36 (expected 36)
Total judges in file : 7 (expected 5)

  ⚠️ claude-3-5-haiku-20241022           | combos: 7/36 | null rows: 35
       MISSING: claude-sonnet-4-6 | best_values | chain_of_thought
       MISSING: claude-sonnet-4-6 | best_values | generated_knowledge
       MISSING: claude-sonnet-4-6 | best_values | zero_shot
       MISSING: claude-sonnet-4-6 | no_context | chain_of_thought
       MISSING: claude-sonnet-4-6 | no_context | theoretical_anchoring
       MISSING: claude-sonnet-4-6 | no_context | zero_shot
       MISSING: deepseek-v3.2 | best_values | chain_of_thought
       MISSING: deepseek-v3.2 | best_values | generated_knowledge
       MISSING: deepseek-v3.2 | best_values | zero_shot
       MISSING: deepseek-v3.2 | no_context | chain_of_thought
       MISSING: deepseek-v3.2 | no_context | theoretical_anchoring
       MISSING: deepseek-v3.2 | no_context | zero_shot
       MISSING: gemini-3.1-pro-preview | all_logs | generated_knowledg